In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import os
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

DEBIAS_LAYER = 15
BASE_ACTS_PATH = f"data/activations/base_acts_train_layer_{DEBIAS_LAYER}.csv"
CAVS_DIR = "models/cavs"

os.makedirs(CAVS_DIR, exist_ok=True)

In [3]:
print("Loading baseline activations for CAV training...")
acts_df = pd.read_csv(BASE_ACTS_PATH)
labels_all = acts_df['label'].values
acts_all = acts_df.drop(columns=['label']).values
acts_train, acts_test, labels_train, labels_test = train_test_split(acts_all, labels_all, test_size=0.2, random_state=42, stratify=labels_all)
print(f"Train set: {acts_train.shape}, Test set: {acts_test.shape}")

Loading baseline activations for CAV training...
Train set: (1600, 1024), Test set: (400, 1024)


In [4]:
# Method 1: Diff Means CAV
mean_bald = acts_train[labels_train == 1].mean(axis=0)
mean_not_bald = acts_train[labels_train == 0].mean(axis=0)
cav_diff_means = mean_bald - mean_not_bald
cav_diff_means /= np.linalg.norm(cav_diff_means)

# Evaluate Diff Means on Train and Test sets
midpoint = (mean_bald + mean_not_bald) / 2.0
threshold = np.dot(midpoint, cav_diff_means)
preds_dm_train = (acts_train @ cav_diff_means > threshold).astype(int)
preds_dm_test = (acts_test @ cav_diff_means > threshold).astype(int)
print("Classification Report for Diff Means CAV (Train Set):")
print(classification_report(labels_train, preds_dm_train))
print("\nClassification Report for Diff Means CAV (Test Set):")
print(classification_report(labels_test, preds_dm_test))

# Save Diff Means CAV
np.save(os.path.join(CAVS_DIR, f"cav_dm_layer_{DEBIAS_LAYER}.npy"), cav_diff_means)

Classification Report for Diff Means CAV (Train Set):
              precision    recall  f1-score   support

           0       0.87      0.78      0.83       800
           1       0.80      0.89      0.84       800

    accuracy                           0.83      1600
   macro avg       0.84      0.83      0.83      1600
weighted avg       0.84      0.83      0.83      1600


Classification Report for Diff Means CAV (Test Set):
              precision    recall  f1-score   support

           0       0.84      0.74      0.79       200
           1       0.77      0.86      0.81       200

    accuracy                           0.80       400
   macro avg       0.81      0.80      0.80       400
weighted avg       0.81      0.80      0.80       400



In [5]:
# Method 2: Logistic Regression CAV
lr_cav = LogisticRegression(C=0.1, max_iter=1000)
lr_cav.fit(acts_train, labels_train)
cav_linear_reg = lr_cav.coef_[0]
cav_linear_reg /= np.linalg.norm(cav_linear_reg)

# Evaluate LogReg on Train and Test sets
print("Classification Report for Logistic Regression CAV (Train Set):")
preds_lr_train = lr_cav.predict(acts_train)
print(classification_report(labels_train, preds_lr_train))
print("\nClassification Report for Logistic Regression CAV (Test Set):")
preds_lr_test = lr_cav.predict(acts_test)
print(classification_report(labels_test, preds_lr_test))

# Save Logistic Regression model and CAV vector
pickle.dump(lr_cav, open(os.path.join(CAVS_DIR, f"lr_model_layer_{DEBIAS_LAYER}.pkl"), "wb"))
np.save(os.path.join(CAVS_DIR, f"cav_lr_layer_{DEBIAS_LAYER}.npy"), cav_linear_reg)

print("CAVs successfully generated and saved.")

Classification Report for Logistic Regression CAV (Train Set):
              precision    recall  f1-score   support

           0       0.95      0.90      0.92       800
           1       0.91      0.95      0.93       800

    accuracy                           0.93      1600
   macro avg       0.93      0.93      0.93      1600
weighted avg       0.93      0.93      0.93      1600


Classification Report for Logistic Regression CAV (Test Set):
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       200
           1       0.86      0.94      0.89       200

    accuracy                           0.89       400
   macro avg       0.89      0.89      0.89       400
weighted avg       0.89      0.89      0.89       400

CAVs successfully generated and saved.
